# DATASCI 267 - GenAI - Assignment 5

### Overview

In Assignment 5 you will create and test a RAG system yourself as a proof of concept and then write a corresponding business report on your findings.

The overall scenario is as follows:

You work at a tech company that is looking for new ways to organize their question answering and search capabilities to accelerate both engineering activity and the marketing team's production.  Founded in 2018, your company has grown rapidly to 300 engineers across distributed teams in San Francisco, Austin, and remotely, plus a 40-person marketing organization split between demand generation, content, product marketing, and customer education. The company also wants to roll out new GenAI-based products, so a lot of the questions will center around Generative AI concepts. Product releases are done quarterly.

As a Senior ML Engineer on your company's newly formed AI Infrastructure team, you've been tapped to lead a four-week proof-of-concept (mini-POC) evaluating whether a Retrieval-Augmented Generation (RAG) system could address the challenges and support both the engineering and marketing organizations. Your executive sponsor is the VP of Engineering, who has secured buy-in from the CMO and allocated a small budget for this exploratory work.

Your POC needs to demonstrate:

**Engineering use case**: Can RAG effectively answer technical questions about GenAI concepts, internal system architecture, and implementation details by retrieving from diverse documentation sources?

**Marketing use case**: Can RAG help marketing staff quickly find accurate, approved messaging about GenAI features, competitive positioning, and technical capabilities to accelerate content production?

**Practical viability**: What are the implementation complexities, accuracy levels, and user experience considerations? Is this worth a larger investment?

Success will be measured by demonstrated accuracy compared to ground truth answers and a clear recommendation on whether to proceed with a full implementation.  If recommended, then a qualitative test will be conducted to get feedback from a small pilot group (5 engineers, 3 marketers).


You will receive a gold dataset with 'good' responses to questions from marketing and engineering teams. You need to develop metric(s) that help you to evaluate how well your RAG system performs relative to the gold data. You should work with the tunable hyperparameters of the setup (LLM, chunking, embeddings, ...) for your iterations.

You will also need to write up your findings as a short 4-5 page proposal.

(See instructions throughout this notebook.)

So overall, the goals of this assignment are for you to:

*  Implement a RAG system using LangChain
*  Be able to formulate metric(s) that you may want to choose as your evaluation of the degree to which your system replicates gold answers (labeled data) that we will provide.
* Try out various hyper-parameters, settings, and other techniques to see which configuration works the best (given your chosen metric)  
* Write a comprehensive evaluation report, which also includes risks and limitations (and a lot more)

The notebook is organized as follows:

1. Set-Up

2. Base RAG components

    We will provide a base LangChain-based framework for you to use as inspiration for your RAG system. The components we’ll need include:  

  - 2.1 Text Embeddings    
  - 2.2 Text Chunking   
  - 2.3 The Vector DB & Semantic Search  
  - 2.4 The Language Model   
  - 2.5 Testing the LLM in a LangChain Chain   
  - 2.6. Setting up a simple RAG Chain     


3. Using RAG  

    We will provide a collection of documents for you to use for your RAG system. We will also provide a test of questions with "Gold" answers.  

  - 3.1 Loading of Data  
  - 3.2 Test Queries
  - 3.3 Running Your Implemented Model(s) and Your Experiments


4.  Evaluations

    Here, you will develop your strategy and conduct your evaluations of your RAG system

  - 4.1 Metrics
  - 4.2 Best Runs with Metrics


5. Final Results

    In this section, you will answer some questions and provide a link to your final report

  - 5.1 Model Specifications
  - 5.2 Three Test Questions (and Results)
  - 5.3 Five Other Questions
  - 5.4 Link To Your 4-5 Page Final POC Report and Recommendation

RULES:  

* You must only use the language models we specify here for the functions we specify
* You must only use the embedding models we specify here for the functions we specify
* You must use the vector store we specify here
* You must only use the document collection we provide. And they ALL must be in your vector store.   
* Apart from the provided specifications, you are free to experiment with a wide variety of hyperparameters (chunk sizes, prompts, etc.), langchain components (text splitters, retrievers, etc.)


**To run this notebook** you should copy it to your personal Colab Pro Google account by uploading it into your Google Drive. From there you can open it as a Colab notebook and run it.  Note: it needs a T4 GPU to run when you are working with all our data.  You can run sections 1 and 2 without the GPU to orient yourself to how LangChain RAG systems work. You should also be able to run it in a free Colab notebook.

NOTES:
* The Open Source Model we use is not trained for safety. So unsafe answers could be returned.


===========================================================================================================

## 1. Setup

We will first install a number of libraries and import what we will need.





In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [ ]:
#pip install --upgrade jupyter_client

In [ ]:
%%capture
!pip install -q -U transformers
!pip install -q -U datasets loralib sentencepiece
!pip install -q bitsandbytes accelerate
!pip install -q -U langchain
!pip install  -q -U langchain-core
!pip install -q -U langchain-classic
!pip install -q -U einops
!pip install -q -U faiss-gpu
#!pip install -q -U langchain_community
#!pip install -q -U chromadb bs4 qdrant-client
!pip install -q -U langchainhub
!pip install -q -U langchain-huggingface
!pip install -q -U langchain-cohere
!pip install -q -U  wikipedia
!pip install -q -U  arxiv
!pip install -q -U  pymupdf
!pip install -q -U xmltodict
!pip install -q -U cohere
!pip install -q -U langchain-qdrant  # updated instead of langchain_community


In [ ]:
%%capture
#In case we want to know our installed transformers library version
!pip list | grep transformers
!pip list | grep accelerate
!pip list | grep langchain

In [ ]:
%%capture
import torch
import os
import bs4
import json
import numpy as np
import time
import pandas as pd
import csv
import re

os.environ['USER_AGENT'] = 'MyLangChainApp/1.0'

from pprint import pprint

import locale

from transformers import AutoTokenizer , AutoModelForCausalLM
from transformers import pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

from langchain_cohere import ChatCohere

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser


from langchain_text_splitters import CharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_classic import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma
#from langchain_community.vectorstores import Qdrant
from langchain_qdrant import QdrantVectorStore   #  importing this  instead
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams


from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.utils.math import cosine_similarity

from langchain_community.document_loaders import ArxivLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.document_loaders import OnlinePDFLoader
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import PubMedLoader



from google.colab import userdata

In [ ]:

import langchain
print(langchain.__version__)
import langchainhub

1.1.2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
locale.getpreferredencoding = lambda: "UTF-8"

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
%%capture
!pip install -U sentence_transformers

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')


# Define your directory
output_folder = "/content/drive/MyDrive/MIDS_267/final_project/output/"
data_folder = "/content/drive/MyDrive/MIDS_267/final_project/data/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Add your keys from the secret store (do **NOT** print them out or leave them exposed as plaintext in your notebook!):

In [ ]:

#COHERE_API_KEY = userdata.get('COHERE_API_KEY')
#CLAUDE_API_KEY = userdata.get('CLAUDE_API_KEY')
#OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN')
# Set the environment variable
#os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
#os.environ["COHERE_API_KEY"] = COHERE_API_KEY
#os.environ["CLAUDE_API_KEY"] = CLAUDE_API_KEY

In [ ]:
# skipping notebook steps from section 2 for experiments to save time
AUDIENCE_TYPE_RESEARCH = 'research'
AUDIENCE_TYPE_MARKETING = 'marketing'

## 2. Building the Components of our RAG System

Let us introduce and test the base components of our RAG system. We will largely use the Hugging Face and LangChain libraries.



### 2.1 The Embedding Model

We will need to represent text (pieces) as vectors. For this, we will use the [sentence_transformer](https://sbert.net/docs/sentence_transformer/pretrained_models.html) architecture.



**NOTE:** The embedding models you can use are: 'all-mpnet-base-v2', 'all-MiniLM-L12-v2', 'multi-qa-mpnet-base-dot-v1', 'all-distilroberta-v1', and 'multi-qa-distilbert-cos-v1 '



In [ ]:

#embedding_models = ['all-mpnet-base-v2', 'all-MiniLM-L12-v2', 'multi-qa-mpnet-base-dot-v1', 'all-distilroberta-v1',  'multi-qa-distilbert-cos-v1 ']
optimal_embedding_model = [ 'multi-qa-mpnet-base-dot-v1'] # based on embedding experiment





class SafeNormalizedEmbeddings(HuggingFaceEmbeddings):
    """Embeddings with safe normalization"""

    def embed_documents(self, texts):
        embeddings = super().embed_documents(texts)
        return [self._safe_normalize(emb) for emb in embeddings]

    def embed_query(self, text):
        embedding = super().embed_query(text)
        return self._safe_normalize(embedding)

    def _safe_normalize(self, embedding):
        emb_array = np.array(embedding, dtype=np.float64)

        if np.any(np.isnan(emb_array)) or np.any(np.isinf(emb_array)):
            return np.zeros_like(emb_array).astype(np.float32).tolist()

        try:
            norm = np.linalg.norm(emb_array)
        except:
            return np.zeros_like(emb_array).astype(np.float32).tolist()

        if norm == 0 or np.isnan(norm) or np.isinf(norm):
            return np.zeros_like(emb_array).astype(np.float32).tolist()

        normalized = (emb_array / norm).astype(np.float32)
        return normalized.tolist()








In [ ]:
# Initialize the vector store

def create_qvector_store(normalized_embeddings, embedding_dim):

    # Initialize client
    client = QdrantClient(location=":memory:")


    # Create the collection first
    client.create_collection(
        collection_name="test",
        vectors_config=VectorParams(
            size=embedding_dim,  # embedding vector dimension
            distance=Distance.COSINE
        )
    )

    # Use QdrantVectorStore instead of Qdrant
    qdrant_vectorstore = QdrantVectorStore(
        client=client,
        collection_name="test",
        #embedding=base_embeddings,
        embedding=normalized_embeddings,
        #text_key="text"
    )
    return qdrant_vectorstore




### 2.4. The LLM

We will use one Open Source Model ("mistralai/Mistral-7B-Instruct-v0.3") and one Proprietery Model (Cohere) for our tests. Let's first set up the OS model:

In [ ]:
#Quantization config
quantization_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_use_double_quant=True,
      bnb_4bit_compute_dtype=torch.bfloat16
    )

LLM_MISTRAL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

def load_model_and_tokenizer(model_name):

    if model_name == 'mistralai/Mistral-7B-Instruct-v0.3':
        llm_mistral_model = AutoModelForCausalLM.from_pretrained(
        "mistralai/Mistral-7B-Instruct-v0.3",
        dtype=torch.float32,
        device_map='auto',
        quantization_config=quantization_config
        )

        llm_mistral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
        llm_mistral_tokenizer.pad_token = llm_mistral_tokenizer.eos_token
        return llm_mistral_model, llm_mistral_tokenizer
    else:
        return None, None


In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
output_parser = StrOutputParser()


Now let's work with an actual document collection.  We will work with four types of documents:

* A few papers from the ArXiv on RAG and NLP
* A few blogs from Lily Weng that talk about Open Domain Question Answering and related topics
* A number of Wikipedia articles on that topic

To make testing easier  we'll define a global record number so we can trace back to see which chunk came from which specific document.


In [ ]:
#assign a unique number to each document we ingest
global_doc_number = 1

First we'll grab some papers from ArXiv.  We'll grab the pdf files and get all of the pages as separate documents.

In [ ]:
arxiv_numbers = ('2005.11401', '2104.07567', '2104.09864', '2105.03011', '2106.09685', '2203.15556', '2203.02155', '2211.09260', '2211.12561',
                 '2212.09741', '2305.14314', '2305.18290', '2306.15595', '2309.08872', '2309.15217', '2310.06825', '2310.11511',
                 '2311.08377', '2312.05708', '2401.06532', '2401.17268', '2402.01306', '2402.19473', '2406.04744',
                 '2312.10997', '2410.12812', '2410.15944', '2404.00657',
                 )

In [ ]:
all_arxiv_pages = []

#loop through the papers
for identifier in arxiv_numbers:
    # Construct URL using the arXiv unique identifier
    arx_url = f"https://arxiv.org/pdf/{identifier}.pdf"

    # Extract pages from the document and add them to the list of pages
    arx_loader = PyMuPDFLoader(arx_url)
    arx_pages = arx_loader.load()
    for page_num in range(len(arx_pages)):
        page = arx_pages[page_num]
        #CHANGED
        page.metadata['page_num'] = page_num
        page.metadata['doc_num'] = global_doc_number
        page.metadata['doc_source'] = "ArXiv"
        all_arxiv_pages.append(page)


    global_doc_number += 1

How many docs did we get?  Is that the correct number? And what is the content?

In [ ]:
num_pages = len(all_arxiv_pages)
num_docs = global_doc_number - 1

print(f"{num_docs} documents in total")
print(f"{num_pages} pages in total")

28 documents in total
598 pages in total


In [ ]:
all_arxiv_pages[5].page_content[:150]  # all pages of the Document content

'Table 1: Open-Domain QA Test Scores. For TQA,\nleft column uses the standard test set for Open-\nDomain QA, right column uses the TQA-Wiki\ntest set. See'

In [ ]:
page = all_arxiv_pages[5]
page.metadata


{'producer': 'pdfTeX-1.40.21',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2021-04-13T00:48:38+00:00',
 'source': 'https://arxiv.org/pdf/2005.11401.pdf',
 'file_path': 'https://arxiv.org/pdf/2005.11401.pdf',
 'total_pages': 19,
 'format': 'PDF 1.5',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2021-04-13T00:48:38+00:00',
 'trapped': '',
 'modDate': 'D:20210413004838Z',
 'creationDate': 'D:20210413004838Z',
 'page': 5,
 'page_num': 5,
 'doc_num': 1,
 'doc_source': 'ArXiv'}

Now we need to split the docs into chunks.  LangChain provides a couple of ways to do that.  We'll use for now the `RecursiveCharacterTextSplitter`.

Let's add the vectors to the datastore and see whether we can retrieve a nearest neighbor to a query. Let's look at the second closest match:

Next, let's get some information from Wikipedia on our main topic -- Gen AI.  LangChain provides a DocumentLoader that accesses the Wikipedia API.

In [ ]:
wiki_docs_1 = WikipediaLoader(query="Generative Artificial Intelligence", load_max_docs=4).load()
for idx, text in enumerate(wiki_docs_1):
    wiki_docs_1[idx].metadata['doc_num'] = global_doc_number
    wiki_docs_1[idx].metadata['doc_source'] = "Wikipedia"

    global_doc_number += 1

print('Number of documents: ', len(wiki_docs_1))

wiki_docs_2 = WikipediaLoader(query="Information Retrieval", load_max_docs=4).load()
for idx, text in enumerate(wiki_docs_2):
    wiki_docs_2[idx].metadata['doc_num'] = global_doc_number
    wiki_docs_2[idx].metadata['doc_source'] = "Wikipedia"

    global_doc_number += 1

print('Number of documents: ', len(wiki_docs_2))

wiki_docs_3 = WikipediaLoader(query="Large Language Models", load_max_docs=4).load()
for idx, text in enumerate(wiki_docs_3):
    wiki_docs_3[idx].metadata['doc_num'] = global_doc_number
    wiki_docs_3[idx].metadata['doc_source'] = "Wikipedia"

    global_doc_number += 1

print('Number of documents: ', len(wiki_docs_3))

wiki_docs_4 = WikipediaLoader(query="Retrieval Augmented Generation", load_max_docs=4).load()
for idx, text in enumerate(wiki_docs_4):
    wiki_docs_4[idx].metadata['doc_num'] = global_doc_number
    wiki_docs_4[idx].metadata['doc_source'] = "Wikipedia"

    global_doc_number += 1

print('Number of documents: ', len(wiki_docs_4))

web_loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2020-10-29-odqa/",
               "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
               "https://lilianweng.github.io/posts/2018-06-24-attention/",
               "https://lilianweng.github.io/posts/2023-06-23-agent/",
               "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
               "https://lilianweng.github.io/posts/2024-07-07-hallucination/"),

    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

web_documents = web_loader.load()

for idx, text in enumerate(web_documents):
    web_documents[idx].metadata['doc_num'] = global_doc_number
    web_documents[idx].metadata['doc_source'] = "WWW"
    global_doc_number += 1

print('Number of documents: ', len(web_documents))


Number of documents:  4
Number of documents:  4
Number of documents:  4
Number of documents:  4
Number of documents:  6


### 3.3 The Test Data

You will want to test the system that you (will) have built. Below we give you a validation set that you could take as labeled data (imagine, your user personas would have had these questions and deemed the answers to be good). We also will give you a test set that only contains questions. (This is the set that we will use to get a feel for how well your RAG system corresponds to our Gold model).


In [ ]:

#validation_questions_answers


# read rag_reference_gold from saved file

gold_df = pd.read_csv(f'{data_folder}rag_reference_gold.csv')
print(gold_df.shape)
print(gold_df.head())

# read test_questions from saved file
test_df = pd.read_csv(f'{data_folder}test_questions.csv')
print(test_df.shape)
print(test_df.head())

(78, 4)
   question_num                                           question  \
0             0  What defines a large language model in the con...   
1             1  How do large language models like GPT-3 become...   
2             2  What are some of the architectures used in bui...   
3             3  Can you name some notable large language model...   
4             7  What licensing terms are associated with sourc...   

                                gold_answer_research  \
0  A large language model in the context of natur...   
1  Large language models like GPT-3 become capabl...   
2  Some common architectures used in building art...   
3  Some notable large language models include Mis...   
4  Source-available models like Mistral AI's lang...   

                               gold_answer_marketing  
0  A large language model (LLM) is a language mod...  
1  Large language models like GPT-3 become capabl...  
2  The architectures used in building artificial ...  
3  Some notabl

#### Question Review
Reviewing the documents loaded into the vector store, the following main sources and categories emerge:

source|topic|
|--|--|
|arxiv|fine tuning, RAG|
|wiki| RAG, GenAI, LLM, Info Retrieval|
|WWW| Agent, Hallucination, adversarial attck LLM, attention, prompt engineering, odqa|

So we will try to categorize the sample questions in similar categories by source to help us select a sample to run experiments with.

|topic|question num|www|wiki|arxiv|
|--|--|--|--|--|
|attention|103, 104, 106, 105, 107, 108, 110|yes|yes||
|prompt engineering| 95|yes|||
|agents|84,85,87,88,89|yes|yes||
|odqa|91,92,93,94|yes|yes||
|LLM|0,1,2,3,7,8,9,11,12,13,16,17,18,19,20,22,23,24,25,27|yes|yes||
|RAG|28,30,33,34,35,36,38,39,43,50,53|yes|yes|yes|
|fine tuning| 55,61,63,73,76|yes|yes|yes|


So, we will create a experiment sample making sure to include questions from each source and each topic that they have significant content.

Selected question list:
|Count|Topic|question num|www|wiki|arxiv|
|--|--|--|--|--|--|
|1|odqa|92|yes|||
|2|prompt engineering| 95| yes||
|3|agents|87|yes||
|4|attention|103|yes||
|5|attention|110|yes|yes|yes|
|6|RAG|28||yes|yes|
|7|RAG| 53||yes|yes|
|8|RAG|33|yes|yes|yes|
|9|fine tuning| 63| yes|yes|yes|
|10|fine tuning|76|yes|yes|yes|
|11|LLM|0||yes|
|12|LLM| 11||yes|
|13|LLM|19||yes|
|14|LLM|27||yes|
|15|LLM|7|||yes|



|question num|question|
|----|--|
|0| What defines a large language model in the context of natural language processing tasks?
|7| What licensing terms are associated with source-available models like Mistral AI's language models?
|11| Which components have allowed large language models to surpass their predecessors?
|19| What is the significant enhancement in Claude 2.1 compared to its previous version?
|27| What is the significance of the accuracy percentage achieved by Chinchilla on the MMLU benchmark?
|28| Why is Chinchilla considered more efficient in terms of computing power for inference and fine-tuning?
|33| What assumptions must be met in order for the reparameterization of reward functions to be applied within the context of Plackett-Luce and Bradley-Terry models?
|53| What are the implications of adding updates on the pretraining data during the fine-tuning phase of model development based on the observed performance of the models?
|63| What are the advantages of applying LoRA to transformer models in terms of computational efficiency during training and deployment?
|87|"What are the main components that complement the central controller in an autonomous agent system?"|
|92|"In what ways can an agent's performance be evaluated and refined over time?"|
|95|"What are the primary techniques involved in steering the behavior of language models without modifying their underlying architectures?"|
|103|"What kind of learning challenges does the attention mechanism address in neural machine translation?"|
|110| "How does Neural Turing Machine (NTM) simulate the infinite memory characteristic of Turing machines?"|





In [ ]:
# create experiment question list
experiment_questions = [0, 7, 11, 19, 27, 28, 33, 53, 63, 76, 87,92, 95, 103, 110]
print("Number of experiment questions:", len(experiment_questions))
# display question number and actual question
for question_num in experiment_questions:
  print(question_num, gold_df[gold_df['question_num']== question_num]['question'].iloc[0])


Number of experiment questions: 15
0 What defines a large language model in the context of natural language processing tasks?
7 What licensing terms are associated with source-available models like Mistral AI's language models?
11 Which components have allowed large language models to surpass their predecessors?
19 What is the significant enhancement in Claude 2.1 compared to its previous version?
27 What is the significance of the accuracy percentage achieved by Chinchilla on the MMLU benchmark?
28 Why is Chinchilla considered more efficient in terms of computing power for inference and fine-tuning?
33 What assumptions must be met in order for the reparameterization of reward functions to be applied within the context of Plackett-Luce and Bradley-Terry models?
53 What are the implications of adding updates on the pretraining data during the fine-tuning phase of model development based on the observed performance of the models?
63 What are the advantages of applying LoRA to transformer

In [ ]:

def get_experiment_questions():
  questions = []
  for num in experiment_questions:
    question = gold_df[gold_df['question_num']== num]['question'].iloc[0]
    questions.append({
        'num': num,
        'question': question
        })
  print(f" Total number of Experiment Questions: {len(questions)}")
  return questions

#get_experiment_questions()

def get_full_validation_questions():

  full_val_questions = []
  for idx, row in gold_df.iterrows():
    num = row['question_num']
    question = row['question']
    full_val_questions.append({
        'num': num,
        'question': question
        })

  print(f" Total number of Validation Questions: {len(full_val_questions)}")
  return full_val_questions

def get_test_questions():
  test_questions = []
  for idx, row in test_df.iterrows():
    num = row['question_num']
    question = row['question']
    test_questions.append({
        'num': num,
        'question': question
        })

  print(f" Total number of Test Questions: {len(test_questions)}")
  return test_questions



### 3.3 Running the RAG System

Let's have a quick look at the validation and test data:

Let's now use the data to ask questions against it. So we need to define our prompt templates, the RAG Chain, etc.

We have two types of User Personas we need to support:

1. The engineers, who require pretty detailed information when they ask questions  
2. The marketing team and supporting staff who also will ask questions around GenAI in order to better understand the products and the field as a whole, but a lot more high level answers would likely be in order

**Below, please build your RAG pipeline including the relevant prompts. This is free form so you will need to create your own cells, text documentation as you need, etc.**


### Setup

In [ ]:
# Documents to split
#print("Loading Documents to split")
documents_list = [all_arxiv_pages, web_documents, wiki_docs_1, wiki_docs_2, wiki_docs_3, wiki_docs_4]



#### Helper Text parsing functions

In [ ]:
def clean_llm_response_extreme(response):
    """
    Clean LLM response to make it compatible with RAGAS parser
    """
    if pd.isna(response):
        return ""

    response = str(response)

    # 1. Remove "RESPONSE:" or "ANSWER:" prefixes
    response = re.sub(r'^(RESPONSE|ANSWER):\s*', '', response, flags=re.IGNORECASE)

    # 2. Remove "REFERENCES:" sections and everything after
    response = re.sub(r'\n\s*REFERENCES?:.*$', '', response, flags=re.DOTALL | re.IGNORECASE)

    # 3. Replace multiple newlines with single space
    response = re.sub(r'\n+', ' ', response)

    # 4. Remove markdown formatting (bold, italic, etc.)
    response = re.sub(r'\*\*([^*]+)\*\*', r'\1', response)  # **bold**
    response = re.sub(r'\*([^*]+)\*', r'\1', response)      # *italic*
    response = re.sub(r'__([^_]+)__', r'\1', response)      # __bold__
    response = re.sub(r'_([^_]+)_', r'\1', response)        # _italic_

    # 5. Normalize numbered lists (convert to simple comma-separated)
    # Replace "1. text 2. text" with "text, text"
    response = re.sub(r'(\d+)\.\s+', '', response)

    # 6. Normalize quotation marks (use consistent double quotes)
    response = response.replace('""', '"')
    response = response.replace('"', '"').replace('"', '"')

    # 7. Remove extra whitespace
    response = ' '.join(response.split())

    # 8. Truncate if too long (RAGAS works better with reasonable lengths)
    max_length = 1500
    if len(response) > max_length:
        # Truncate at sentence boundary if possible
        truncated = response[:max_length]
        last_period = truncated.rfind('.')
        if last_period > max_length * 0.8:  # If we can find a period in last 20%
            response = truncated[:last_period + 1]
        else:
            response = truncated + "..."

    return response.strip()


def clean_llm_response(response):
    """
    Clean LLM response - MINIMAL VERSION
    """
    if pd.isna(response):
        return ""

    response = str(response)

    # 1. Remove "ANSWER:" prefix only
    response = re.sub(r'^(RESPONSE|ANSWER):\s*', '', response, flags=re.IGNORECASE)

    # 2. Remove markdown formatting
    response = re.sub(r'\*\*([^*]+)\*\*', r'\1', response)
    response = re.sub(r'\*([^*]+)\*', r'\1', response)

    # 3. Normalize whitespace
    response = ' '.join(response.split())

    # NO TRUNCATION - let RAGAS see full response!

    return response.strip()

#parse rag output
def parse_rag_output(output):
  """Extract context and response from RAG output"""
  output_str = str(output)
  #print("output string:", output_str)

  context = ""
  response = ""

  # Extract context
  if 'CONTEXT:' in output_str and 'QUESTION:' in output_str:
      context_start = output_str.find('CONTEXT:') + len('CONTEXT:')
      context_end = output_str.find('QUESTION:')
      context = output_str[context_start:context_end].strip()

  # Extract response
  if '[/INST]' in output_str:
      response_start = output_str.find('[/INST]') + len('[/INST]')
      response = output_str[response_start:].strip()

  return context, response








### Embedding model

In [ ]:


# Calculate max token lengths for LLM
def calculate_max_token_lengths(df, rag_audience, tokenizer):
  if rag_audience == AUDIENCE_TYPE_RESEARCH:
    column_name = 'gold_answer_research'
  elif rag_audience == AUDIENCE_TYPE_MARKETING:
    column_name = 'gold_answer_marketing'

  token_lengths = []
  for answer in df[column_name]:
      tokens = tokenizer.encode(answer)
      token_lengths.append(len(tokens))

  # 2. Use percentile to handle outliers
  p95_length = int(np.percentile(token_lengths, 95))
  p99_length = int(np.percentile(token_lengths, 99))

  # 3. Add buffer for generation flexibility
  max_new_tokens = int(p95_length * 1.2)  # 20% buffer

  print(f"Answer token length stats:")
  print(f"  Mean: {np.mean(token_lengths):.0f}")
  print(f"  Median: {np.median(token_lengths):.0f}")
  print(f"  95th percentile: {p95_length}")
  print(f"  99th percentile: {p99_length}")
  print(f"  Max: {max(token_lengths)}")
  print(f"  Recommended max_new_tokens for {column_name}: {max_new_tokens}")
  return max_new_tokens



### LLM for Text Generation

In [ ]:
# LLM


def configure_llm_pipe(max_new_tokens, llm_model, model_name, temp, top_p, do_sample, repetition_penalty):
    print(f"Configuring LLM pipe for {model_name}")

    print("max_new_tokens", max_new_tokens)

    #TEMPERATURE = 0.6
    #TOP_P = 0.95
    #DO_SAMPLE = True
    #REPETITION_PENALITY = 1.2

    mistral_pipe = pipeline(
        "text-generation",
        model=llm_model,  # initialized earlier
        tokenizer=llm_tokenizer, # initialized earlier
        max_new_tokens=max_new_tokens,
        temperature=temp,
        top_p=top_p,
        do_sample=do_sample,
        repetition_penalty=repetition_penalty
    )
    mistral_pipe.model.config.pad_token_id = mistral_pipe.model.config.eos_token_id

    # wrapping the Hugging Face pipeline into a LangChain object

    mistral_llm_lc = HuggingFacePipeline(pipeline=mistral_pipe)
    return mistral_llm_lc

def configure_cohere_llm(max_new_tokens,  temp, top_p, do_sample, repetition_penalty):
  cohere_llm_lc = None
  cohere_chat_model = ChatCohere(
        cohere_api_key=COHERE_API_KEY,
        model="command-r-plus",  # Best model
        temperature=temp,
        max_tokens=max_new_tokens,
        p=top_p,
        frequency_penalty=frequency_penalty,
    )


  return cohere_chat_model





### RAG PROMPT

# 1. Add context relevance indicator
"First, rate how relevant the context is to the question (High/Medium/Low), then answer."

# 2. Add source attribution
"When possible, reference which part of the context supports your answer."

# 3. For numerical/technical accuracy
"Include specific numbers, parameters, or measurements from the context when available."

# 4. Control verbosity
"Answer in exactly 3-5 sentences. Be concise but technically complete."



|Improvement|Why It Helps|
|--|--|
|Role definition| "You are a technical assistant" primes the model for technical responses|
|Step-by-step reasoning| Reduces hallucination by forcing context analysis first|
|Explicit constraints| "ONLY" and "EXCLUSIVELY" reinforce grounding|
|Partial answer guidance| Handles edge cases where context is incomplete|
|Removed negative instructions| "Do NOT include markdown" → removed (can confuse model)|
|Clearer structure| Separates CONTEXT, QUESTION, INSTRUCTIONS, ANSWER|
|Few-shot example| Shows expected output format and depth|


ImprovementWhy It Helps Marketing PromptsRole: "communications/marketing assistant"Primes model for business language"Translate technical → business benefits"Explicitly guides jargon removalFew-shot exampleShows expected tone and style"What it does + why it matters"Focuses on value, not featuresBenefit-first structureAligns with marketing communication styleRemoved negative phrasingClearer instructions work betterPartial answer handlingAddresses incomplete context gracefully


|Aspect|Research Prompt|Marketing Prompt|
|-|-|-|
|Length|3-5 sentences|1-2 sentences|
|Focus|Implementation details|Benefits & value|
|Language|Technical terminology|Simple, jargon-free|
|Tone|Precise, detailed|Clear, engaging|
|Content|Methods, algorithms, specs|Capabilities, advantages

In [ ]:
#  RAG prompt

def get_rag_prompt_template(audience_type):

  rag_prompt_research = """[INST]Answer the following question using ONLY the provided context. Generate answers for Research Engineering Team audience:

      CONTEXT:
      {context}

      QUESTION:
      {question}

      Generate a response with:
      - "ANSWER": Detailed technical answer (3-5 sentences) for engineers including:
        * Technical implementation details
        * Methodologies and algorithms
        * Mathematical concepts if applicable
        * Full technical terminology


      IMPORTANT:
      - Base ANSWER ONLY on the CONTEXT provide
      - If the answer isn't in the context, say: 'I don't have enough information to answer this accurately.'
      - Do NOT include markdown formatting, backticks, or any text outside the ANSWER:[/INST]"""

  rag_prompt_marketing = """[INST]Answer the following question using ONLY the provided context. Generate answers for Marketing Team audience:

      CONTEXT:
      {context}

      QUESTION:
      {question}

      Generate a response with:
      - "ANSWER": Simplified answer (1-2 sentences) for marketing including:
        * Key capabilities and benefits
        * Clear, business-friendly language
        * Minimal jargon

      IMPORTANT:
      - Base answer ONLY on the CONTEXT provided
      - If the answer isn't in the context, say: 'I don't have enough information to answer this accurately.'
      - Do NOT include markdown formatting, backticks, or any text outside the ANSWER:[/INST]"""

  if audience_type == AUDIENCE_TYPE_RESEARCH:
    rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt_research)
  elif audience_type == AUDIENCE_TYPE_MARKETING:
    rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt_marketing)
  return rag_prompt_template



In [ ]:
#Research prompt options
#Enhanced structure with reasoning


research_prompt_1 =  """[INST]You are a technical assistant for a Research Engineering Team. Answer the question using ONLY the provided context.

CONTEXT:
{context}

QUESTION:
{question}

INSTRUCTIONS:
1. First, identify the relevant information in the context
2. Then, provide a detailed technical answer (3-5 sentences) that includes:
  - Technical implementation details and specifications
  - Methodologies, algorithms, or frameworks mentioned
  - Mathematical concepts or formulas if applicable
  - Proper technical terminology

RULES:
- Use ONLY information from the CONTEXT above
- If information is insufficient, state: "Based on the available context, I cannot fully answer this question."
- If only partial information exists, provide what you can and note the limitations
- Be precise and avoid speculation beyond the context

ANSWER:[/INST]"""

#Chain of thought style
research_prompt_2 =  """[INST]You are a technical research assistant. Using ONLY the context provided, answer the question for a Research Engineering audience.

CONTEXT:
{context}

QUESTION:
{question}

Think step-by-step:
1. What relevant information is in the context?
2. What technical details, methods, or specifications are mentioned?
3. How do these relate to the question?

Then provide a detailed technical answer (3-5 sentences) with:
- Implementation details and technical specifications
- Algorithms, methodologies, or mathematical concepts
- Appropriate technical terminology

If the context lacks sufficient information, explicitly state what is missing.

ANSWER:[/INST]"""

#With Few Shot Example
research_prompt_3 =  """[INST]You are a technical assistant for a Research Engineering Team. Answer questions using ONLY the provided context.

EXAMPLE:
Context: "The transformer architecture uses self-attention mechanisms with O(n²) complexity. The model employs 12 attention heads with 768-dimensional embeddings."
Question: "What is the computational complexity of the attention mechanism?"
Answer: The transformer's self-attention mechanism has O(n²) computational complexity, where n is the sequence length. The architecture implements multi-head attention with 12 parallel attention heads operating on 768-dimensional embeddings, allowing the model to attend to different representation subspaces simultaneously.

NOW ANSWER THIS:

CONTEXT:
{context}

QUESTION:
{question}

Provide a detailed technical answer (3-5 sentences) including implementation details, methodologies, and technical terminology. Use ONLY the context provided. If information is insufficient, state what is missing.

ANSWER:[/INST]"""

# Enhanced prompts for Marketing

# Enhanced Structure with Business Focus
marketing_prompt_1 =  """[INST]You are a communications assistant for a Marketing Team. Answer the question using ONLY the provided context in clear, business-friendly language.

CONTEXT:
{context}

QUESTION:
{question}

INSTRUCTIONS:
1. First, identify the key points relevant to the question
2. Then, provide a clear, concise answer (1-2 sentences) that includes:
   - Key capabilities and business benefits
   - Value proposition or competitive advantages
   - Simple, jargon-free language

RULES:
- Use ONLY information from the CONTEXT above
- Translate technical concepts into business benefits
- If information is insufficient, state: "Based on the available information, I cannot fully answer this question."

ANSWER:[/INST]"""

# Benefit-Focused with Reasoning
marketing_prompt_2 =  """[INST]You are a marketing communications assistant. Using ONLY the context provided, answer the question for a non-technical Marketing Team.

CONTEXT:
{context}

QUESTION:
{question}

Think step-by-step:
1. What key information in the context relates to the question?
2. What are the business benefits or value points?
3. How can this be explained simply?

Then provide a clear answer (1-2 sentences) that:
- Highlights capabilities and benefits
- Uses simple, accessible language
- Avoids technical jargon (or explains it simply)

If the context lacks sufficient information, state what is missing.

ANSWER:[/INST]"""

#Few Shot Example
marketing_prompt_3 =  """[INST]You are a marketing assistant who translates technical information into business-friendly language. Answer questions using ONLY the provided context.

EXAMPLE:
Context: "The system uses transformer-based NLP with 95% accuracy for sentiment analysis, processing 10,000 queries per second with sub-100ms latency."
Question: "What can this system do for customer feedback analysis?"
Answer: Our solution analyzes customer sentiment with 95% accuracy, enabling real-time insights from thousands of feedback responses instantly—helping you understand customer needs faster than ever.

NOW ANSWER THIS:

CONTEXT:
{context}

QUESTION:
{question}

Provide a clear, benefit-focused answer (1-2 sentences) using simple language. Focus on what it does and why it matters. Use ONLY the context provided.

ANSWER:[/INST]"""

# structured with audience awareness
marketing_prompt_4 =  """[INST]You are helping a Marketing Team understand technical capabilities. Answer in simple, benefit-focused language using ONLY the provided context.

CONTEXT:
{context}

QUESTION:
{question}

RESPONSE GUIDELINES:
- Keep it brief: 1-2 clear sentences
- Focus on "what it does" and "why it matters"
- Replace jargon with everyday language
- Emphasize benefits, capabilities, or value

CONSTRAINTS:
- Base your answer EXCLUSIVELY on the context above
- Do not add technical details not in the context
- If context is incomplete, state: "The available information covers [X] but doesn't address [Y]"

ANSWER:[/INST]"""

#action  oriented marketing style
marketing_prompt_5 = """[INST]You are a marketing copywriter. Transform technical context into compelling, easy-to-understand answers for a Marketing Team.

CONTEXT:
{context}

QUESTION:
{question}

YOUR TASK:
Write a clear, engaging answer (1-2 sentences) that:
- Leads with the benefit or capability
- Uses active, confident language
- Avoids technical jargon
- Sounds natural and conversational

IMPORTANT:
- Use ONLY facts from the CONTEXT
- If you cannot answer from the context, say so clearly

ANSWER:[/INST]"""

marketing_prompt_5_improved =  """[INST]You are a marketing communications assistant.
Answer the question using ONLY the provided context, translated into
business-friendly language.

CONTEXT:
{context}

QUESTION:
{question}

INSTRUCTIONS:
1. First, identify the key facts in the context relevant to the question
2. Then, write a clear answer (1-2 sentences) that:
   - Highlights the capability or benefit
   - Uses simple, jargon-free language
   - Sounds natural and conversational

RULES:
- Use ONLY facts explicitly stated in the CONTEXT above
- Do not infer, embellish, or add information not present
- If information is insufficient, state: "Based on the available context,
  I cannot fully answer this question."
- Be accurate first, engaging second

ANSWER:[/INST]"""

marketing_prompt_list = [marketing_prompt_1, marketing_prompt_2, marketing_prompt_3, marketing_prompt_4, marketing_prompt_5]

research_prompt_list = [research_prompt_1, research_prompt_2, research_prompt_3]




#### FEW Shot Experiment
using Q19 (worst performing question as an example for training)

19: {
    "question": "What is the significant enhancement in Claude 2.1 compared to its previous version?",
    "gold_answer_research": "The significant enhancement in Claude 2.1 compared to its previous version Claude 2 is the expanded context window, which has been increased from 100,000 tokens to 200,000 tokens. This allows Claude 2.1 to process and analyze larger amounts of text, enabling it to provide more comprehensive summaries and assistance with tasks. Additionally, Claude 2.1 has improved its performance in handling complex queries and requests, showcasing advancements in its capabilities for natural language processing and understanding.",
    "gold_answer_marketing": "The significant enhancement in Claude 2.1 compared to its previous version is the expansion of the context window from 9,000 tokens to 100,000 tokens, as well as the ability to upload PDFs and other documents for reading, summarizing, and task assistance."
  },

In [ ]:
marketing_prompt_5_improved =  """[INST]You are a marketing communications assistant.
Answer the question using ONLY the provided context, translated into
business-friendly language.

CONTEXT:
{context}

QUESTION:
{question}

INSTRUCTIONS:
1. First, identify the key facts in the context relevant to the question
2. Then, write a clear answer (1-2 sentences) that:
   - Highlights the capability or benefit
   - Uses simple, jargon-free language
   - Sounds natural and conversational

RULES:
- Use ONLY facts explicitly stated in the CONTEXT above
- Do not infer, embellish, or add information not present
- If information is insufficient, state: "Based on the available context,
  I cannot fully answer this question."
- Be accurate first, engaging second

ANSWER:[/INST]"""








research_prompt_1_2_hybrid =  """[INST]You are a technical research assistant.
Answer the question using ONLY the provided context for a Research Engineering audience.

CONTEXT:
{context}

QUESTION:
{question}

Think step-by-step:
1. What relevant information is in the context?
2. What technical details, methods, or specifications are mentioned?
3. How do these relate to the question?

Then provide a detailed technical answer (3-5 sentences) with:
- Technical implementation details and specifications
- Algorithms, methodologies, or mathematical concepts
- Appropriate technical terminology

RULES:
- Use ONLY information from the CONTEXT above
- Do not speculate or infer beyond what is explicitly stated
- If information is insufficient, state: "Based on the available context,
  I cannot fully answer this question."
- If only partial information exists, provide what you can and note the limitations

ANSWER:[/INST]"""

marketing_prompt_5_2_hybrid = """[INST]You are a marketing communications assistant.
Answer the question using ONLY the provided context, in clear business-friendly language.

CONTEXT:
{context}

QUESTION:
{question}

Think step-by-step:
1. What key facts in the context relate to the question?
2. What is the main capability, benefit, or value?
3. How can this be explained simply without jargon?

Then write a clear answer (1-2 sentences) that:
- Leads with the benefit or capability
- Uses simple, accessible language
- Sounds natural and conversational

RULES:
- Use ONLY facts explicitly stated in the CONTEXT above
- Do not embellish, infer, or add information not present
- If information is insufficient, state: "Based on the available context,
  I cannot fully answer this question."
- Be accurate first, engaging second

ANSWER:[/INST]"""




#  RAG Prompts with Real Few-Shot Examples
# Using Q48 which achieved: MDS=0.8967, Faithfulness=1.0, Answer Relevancy=0.9922
# Uses actual Q48 example that achieved perfect faithfulness (1.0)

research_prompt_48 = """[INST]You are a technical assistant for a Research Engineering Team. Answer questions using ONLY the provided context.

EXAMPLE:
Context: 'stration on a 1-7 Likert scale, and computed an average "demonstration score" for each labeler.
4. Self-assessed ability to identify sensitive speech for different groups. We wanted to select a team of labelers that had collectively were able to identify sensitive content in a broad range of areas. For legal reasons, we can't hire contractors based on demographic criteria. Thus, we had labelers answer the question: "For what topics or cultural groups are you comfortable identifying sensitive speech?" and used this as part of our selection process.

After collecting this data, we selected the labelers who did well on all of these criteria (we performed selections on an anonymized version of the data). Since the fourth criteria is subjective, we ultimately chose labelers subjectively according to these criteria, though we had soft cutoffs at 75% agreement on sensitive speech flagging and comparisons, and a 6/7 demonstration score.

More specifically, from an initial pool of labeler candidates, we selected our training labelers according to the following criteria:
1. Agreement on sensitive speech flagging. We created a dataset of prompts and completions, where some of prompts or completions were sensitive (i.e. anything that could elicit strong negative feelings, whether by being toxic, sexual, violent, judgemental, political, etc.).'

Question: "What criteria were used to select labelers to ensure they can effectively detect and rate sensitive content?"

Answer: "The criteria used to select labelers to effectively detect and rate sensitive content included agreement on sensitive speech flagging, agreement on rankings of model completions, and a sensitive demonstration writing test. Labelers were selected based on their performance in these tests, with soft cutoffs at 75% agreement on sensitive speech flagging and comparisons, and a 6/7 demonstration score. The sensitive speech flagging was determined by creating a dataset of prompts and completions, where some of them were sensitive, and measuring agreement between the researchers' labels and the labelers'. The ranking of model completions was done by having labelers rank the completions by overall quality and measuring their agreement with researcher labels. The sensitive demonstration writing test involved rating each demonstration on a 1-7 Likert scale and computing an average demonstration score for each labeler."

NOW ANSWER THIS:

CONTEXT:
{context}

QUESTION:
{question}

Provide a detailed technical answer (3-5 sentences) including implementation details, methodologies, and technical terminology. Use ONLY information from the context above. If the context does not contain relevant information, state: "The provided context does not contain sufficient information about [topic]."

ANSWER:[/INST]"""



# MARKETING PROMPT
# Uses actual Q48 marketing example

marketing_prompt_48 = """[INST]You are a marketing assistant who translates technical information into business-friendly language. Answer questions using ONLY the provided context.

EXAMPLE:
Context: 'stration on a 1-7 Likert scale, and computed an average "demonstration score" for each labeler.
4. Self-assessed ability to identify sensitive speech for different groups. We wanted to select a team of labelers that had collectively were able to identify sensitive content in a broad range of areas. For legal reasons, we can't hire contractors based on demographic criteria. Thus, we had labelers answer the question: "For what topics or cultural groups are you comfortable identifying sensitive speech?" and used this as part of our selection process.

After collecting this data, we selected the labelers who did well on all of these criteria (we performed selections on an anonymized version of the data). Since the fourth criteria is subjective, we ultimately chose labelers subjectively according to these criteria, though we had soft cutoffs at 75% agreement on sensitive speech flagging and comparisons, and a 6/7 demonstration score.

More specifically, from an initial pool of labeler candidates, we selected our training labelers according to the following criteria:
1. Agreement on sensitive speech flagging. We created a dataset of prompts and completions, where some of prompts or completions were sensitive (i.e. anything that could elicit strong negative feelings, whether by being toxic, sexual, violent, judgemental, political, etc.).'

Question: "What criteria were used to select labelers to ensure they can effectively detect and rate sensitive content?"

Answer: "To select labelers who can effectively detect and rate sensitive content, we evaluated them based on their agreement on sensitive speech flagging, their ability to rank model completions by overall quality, and their performance on a demonstration writing test measuring their ability to respond appropriately to sensitive prompts."

NOW ANSWER THIS:

CONTEXT:
{context}

QUESTION:
{question}

Provide a clear, benefit-focused answer (1-2 sentences) using simple language. Focus on what it does and why it matters. Use ONLY information from the context above. If the context does not contain relevant information, state: "The available information does not cover this topic."

ANSWER:[/INST]"""


#
# MARKETING PROMPT - Q33 FEW-SHOT EXAMPLE
#
# Q33 achieved: MDS=0.9138, Faithfulness=1.0, Answer Relevancy=0.9939
# Good for: Complex technical → Simple 1-sentence answer

marketing_prompt_q33 = """[INST]You are a marketing assistant who translates technical information into business-friendly language. Answer questions using ONLY the provided context.

EXAMPLE:
Context: 'we usually have to impose additional identifiability constraints to achieve any guarantees on the MLE estimates from Eq. 2 [4]. The second lemma states that all reward functions from the same class yield the same optimal policy, hence for our final objective, we are only interested in recovering an arbitrary reward function from the optimal class. We prove the following Theorem in Appendix A.6:
Theorem 1. Under mild assumptions, all reward classes consistent with the Plackett-Luce (and Bradley-Terry in particular) models can be represented with the reparameterization r(x, y) = β log π(y|x)

Specifically, we first take the logarithm of both sides of Eq. 4 and then with some algebra we obtain: r(x, y) = β log πr(y | x) πref(y | x) + β log Z(x). We can apply this reparameterization to the ground-truth reward r∗and corresponding optimal model π∗. Fortunately, the Bradley-Terry model depends only on the difference of rewards between two completions.'

Question: "What assumptions must be met in order for the reparameterization of reward functions to be applied within the context of Plackett-Luce and Bradley-Terry models?"

Answer: "The reparameterization of reward functions can be applied within the Plackett-Luce and Bradley-Terry models under mild assumptions, as long as the reward classes are consistent with these models."

NOW ANSWER THIS:

CONTEXT:
{context}

QUESTION:
{question}

Provide a clear, benefit-focused answer (1-2 sentences) using simple language. Focus on what it does and why it matters. Use ONLY information from the context above. If the context does not contain relevant information, state: "The available information does not cover this topic."

ANSWER:[/INST]"""


In [ ]:
def get_rag_prompt_template_for_prompt(rag_prompt):
  rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)
  return rag_prompt_template

KEY IMPROVEMENTS:

1. EXAMPLE IS GROUNDED IN CONTEXT
    Example answer directly references information IN the example context

2. DEMONSTRATES CORRECT BEHAVIOR
   - Shows LLM how to extract and synthesize info from context
   - Answer format matches the style we want (technical vs simple)

3. HANDLES INSUFFICIENT INFO
   - Explicit instruction on what to say when context lacks info
   - Prevents hallucination on questions where retrieval fails

4. CONSISTENT INSTRUCTIONS
   - Example reinforces the "ONLY use context" rule
   - No contradiction between example and instructions

EXPECTED IMPROVEMENTS:
- Research: Fewer zero Answer Relevancy scores
- Both: Higher Faithfulness (better grounding)
- Both: More consistent performance across questions

What is the significant enhancement in Claude 2.1 compared to its previous version?

improvements in knowledge-intensive tasks after fine-tuning or used with off-the-shelf LMs (Ram
et al., 2023). A more recent work (Luo et al., 2023) instruction-tunes an LM with a fixed number
2

32.3
52.3
37.0
77.0
70.1
80.5
30.4
53.7
29.6
57.9
51.9
GTR-XL (1.5B)
48.0
56.0
41.5
86.1
67.1
77.8
30.2
58.4
32.0
60.4
55.5
INSTRUCTOR-XL (1.5B)
49.3
57.3
44.7
86.6
73.2
83.1
32.0
61.8
34.1
68.6
58.8
Relative gain (%)
+2.7
+2.3
+7.7
+0.6
+9.1
+6.9
+6.0
+5.8
+6.6
+13.6
+5.9
Table 2: Results on the massive text embedding benchmark (MTEB; Muennighoff et al., 2022), Billboard (Kasai
et al., 2022a), and prompt retrieval (Su et al., 2022). The last column averages performance scores over 9 categories

nih_exporter
uspto_backgrounds
pubmed_central
pile_cc
bookcorpus2
stackexchange
opensubtitles
openwebtext2
hackernews
dm_mathematics
arxiv
freelaw
books3
philpapers
github
ubuntu_irc
europarl
gutenberg_pg_19
0.00
0.02
0.04
0.06
0.08
0.10
Decrease in bpb
 compared to Gopher
Figure 5 | Pile Evaluation. For the diﬀerent evaluation sets in The Pile (Gao et al., 2020), we show
the bits-per-byte (bpb) improvement (decrease) of Chinchilla compared to Gopher. On all subsets,
Chinchilla outperforms Gopher.

approach, fine-grained segmentation, and the incorporation of
metadata. Additionally, it incorporates several optimization
methods to streamline the retrieval process [8].

MTEB, Billboard, and prompt retrieval respec-
tively. Specifically, among all task categories, IN-
STRUCTOR (335M) demonstrates large improve-
ments over GTR-Large on the text evaluation
(18.3%), classification (10.1%), and clustering
tasks (8.9%). Particularly noteworthy is INSTRUC-
TOR’s performance compared to the previous state-
of-the-art model, Sent-T5-XXL (58.4 vs. 56.5 on
average), despite the fact that INSTRUCTOR has
one order of magnitude fewer parameters (335M
vs. 4.8B).

### Experiment Questions

When configuring text splitters for different document types, chunk size and overlap should be adjusted based on content characteristics:

## Technical Documents

**Recommended Settings:**
- **Chunk size**: 500-1000 tokens (smaller chunks)
- **Overlap**: 100-200 tokens (20-30% overlap)

**Reasoning:**
- Technical content is dense with specific terminology, code snippets, and precise relationships
- Smaller chunks preserve context for individual concepts, functions, or procedures
- Higher overlap ensures technical dependencies and cross-references aren't lost at chunk boundaries
- Code blocks and formulas should remain intact within chunks

## Regular Documents (Narrative/General Text)

**Recommended Settings:**
- **Chunk size**: 1000-1500 tokens (larger chunks)
- **Overlap**: 100-200 tokens (10-15% overlap)

**Reasoning:**
- Narrative text benefits from more context to maintain story flow or argument development
- Larger chunks capture complete thoughts, paragraphs, or sections
- Lower overlap percentage is sufficient since ideas are more naturally separated
- Less risk of breaking critical dependencies

## Key Considerations

**Technical documents need smaller chunks because:**
- A single line of code or formula can be critical
- Definitions and implementations should stay together
- API references and parameter lists need preservation

**Regular documents work better with larger chunks because:**
- Meaning emerges over longer passages
- Context accumulates gradually
- Paragraph breaks provide natural boundaries


### setup variable text splitter

| Document Type | Splitter | Chunk Size (chars) | Overlap |
|---------------|----------|-------------------|---------|
| API docs | `RecursiveCharacterTextSplitter` | 800-1000 | 200 |
| Code files | `RecursiveCharacterTextSplitter.from_language()` | 600-800 | 150-200 |
| Technical manuals | `MarkdownTextSplitter` | 1000 | 200 |
| Blog posts | `RecursiveCharacterTextSplitter` | 1500 | 150 |
| Books/articles | `RecursiveCharacterTextSplitter` | 1500-2000 | 150-200 |




| Document Type | Chunk Size | Overlap | Reasoning |
|---------------|------------|---------|-----------|
| Technical Blogs | 1000-1200 | 200 | Balance between code snippets and explanations |
| Wikipedia | 1500 | 150 | Encyclopedic content needs more context |
| Technical Papers | 1000 | 250 | Dense content, higher overlap for continuity |

In [ ]:


#RAG chain with variable retriever
def create_rag_chain(rag_audience, retriever, prompt_template, model_langchain):
      rag_prompt_template = get_rag_prompt_template(rag_audience)
      base_rag_chain =(
          {"context": retriever | format_docs,
          "question": RunnablePassthrough()}
          | prompt_template     #rag_prompt_template
          | model_langchain   #mistral_llm_lc/cohere_llm
          | output_parser
      )
      return base_rag_chain


In [ ]:
def save_experiment_results(results,question_nums, questions , experiment_name):
  "save rag output to csv file"
  print(f"\n{'='*60}")
  num_exp = len(results)
  print(f"Number of experiments: {num_exp}")
  print(f"{'='*60}")
  print(f"Saving parsed output to {output_folder}rag_{experiment_name}_outputs.csv")
  with open(f'{output_folder}rag_{experiment_name}_outputs.csv', 'w', newline='', encoding='utf-8') as csvfile:
          writer = csv.writer(csvfile)
          writer.writerow(['embedding_model', 'embedding_dim', 'config_name','chunksize', 'chunkoverlap','num_retrieve',
                           'llm_name', 'llm_temp','llm_top_p','llm_do_sample','llm_repetition_penalty', 'llm_max_new_tokens', 'inference_time',
                           'audience_type', 'question_num','question', 'context', 'response', 'cleaned_response'])
          for idx, result in enumerate(results):
            config_name = result['config_name']
            rag_output = result['rag']
            chunk_size=result['chunk_size']
            chunk_overlap=result['chunk_overlap']
            num_retrieve=result['num_retrieve']
            llm_name=result['llm_name']
            audience_type=result['audience_type']
            embedding_model=result['embedding_model']
            embedding_dim=result['embedding_dim']
            llm_temp=result['llm_temp']
            llm_top_p=result['llm_top_p']
            llm_do_sample=result['llm_do_sample']
            llm_repetition_penalty=result['llm_repetition_penalty']
            llm_max_new_tokens=result['llm_max_new_tokens']
            inference_time=result['inference_time']
            for num,question , output in zip(  question_nums, questions,rag_output):
                # Save parsed output
                context, response = parse_rag_output(output)
                #print(f"\n{'='*60}")
                #print(f"context: {context} \nof response: {response}")
                cleaned_response = clean_llm_response(response)
                writer.writerow([emb_model, embedding_dim, config_name, chunk_size, chunk_overlap, num_retrieve,
                                 llm_name, llm_temp,llm_top_p, llm_do_sample, llm_repetition_penalty, llm_max_new_tokens, inference_time,
                                 audience_type, num, question, context, response, cleaned_response])
            print(f"\n{'='*60}")
            print(f" Saved result for experiment number {idx + 1} of {num_exp}")
            print(f"\n{'='*60}")
  print(f"\n{'='*60}")
  print("Saved all experiment results to file")


def save_rag_output(config, rag_output, chunk_size, chunk_overlap, question_nums, questions,audience_type, llm_name, emb_model, num_retrieve):
  with open(f'{output_folder}rag_{RAG_EXPERIMENT}_outputs.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['chunksize', 'chunkoverlap','audience_type', 'question_num','question', 'context', 'response', 'cleaned_response'])
    # Save parsed output
    for num,question , output in zip(  question_nums, questions,rag_output):
        context, response = parse_rag_output(output)
        cleaned_response = clean_llm_response(response)
        writer.writerow([chunk_size, chunk_overlap, audience_type, num, question, context, response, cleaned_response])

In [ ]:
from datetime import datetime


RAG_EXPERIMENT = "final_test_run_mistral"
#"full_validation_questions"
llm_model,llm_tokenizer,normalized_embeddings = None, None,None
# Selected retriever configuration based on previous experiment on retriever parameters
#retriever_config = {"chunk_size": 512, "overlap": 100, "num_ret": 5, "name": "optimal"}
retriever_config = {"chunk_size": 512, "overlap": 100, "num_ret": 5, "name": "optimal"}
#embedding_models = ['multi-qa-distilbert-cos-v1','all-mpnet-base-v2', 'all-MiniLM-L12-v2', 'multi-qa-mpnet-base-dot-v1', 'all-distilroberta-v1']
embedding_config = [ 'multi-qa-mpnet-base-dot-v1'] # optimal
llm_config = {
    "name": "mistral",
    "model_name": LLM_MISTRAL_NAME,
    "llm_temp" : 0.2,
    "llm_top_p" : 0.95,
    "llm_do_sample" : True,
    "llm_repetition_penalty" : 1.1,
    "llm_max_new_tokens" : 200
}
cohere_config = {
    "name": "cohere",
    "model_name": "Cohere",
    "llm_temp" : 0.2,
    "llm_top_p" : 0.95,
    "llm_do_sample" : True,
    "llm_repetition_penalty" : 1.1,
    "llm_max_new_tokens" : 200
}
use_mistral = True
marketing_prompt_list
research_prompt_list


audience_configs = [

    {
        "name": "research_prompt_48",
        "audience_type": 'research',
        "prompt": research_prompt_48,
    },
    {
        "name": "marketing_prompt_q33",
        "audience_type": 'marketing',
        "prompt": marketing_prompt_q33,
    }

]

# experiment questions
print(f"\n{'='*60}")
print("Getting experiment questions")
#exp_questions = get_experiment_questions()
#full_questions = get_full_validation_questions()
test_questions = get_test_questions()
questions = [item['question'] for item in test_questions]
question_nums = [item['num'] for item in test_questions]

# setup vector store outside the test loop
emb_model = embedding_config[0]
#print(f"Starting experiment for Embedding model: {emb_model}")
print(f"{'='*60}")
normalized_embeddings = SafeNormalizedEmbeddings(model_name=emb_model)
embedding_dim = normalized_embeddings.client.get_sentence_embedding_dimension()
print(f"{'='*60}")
print(f"Embedding dimension: {embedding_dim}")
print(f"{'='*60}")

chunk_size = retriever_config['chunk_size']
chunk_overlap = retriever_config['overlap']
num_retrieve = retriever_config['num_ret']
print(f"\n{'='*60}")
#print(f"Testing: {config_name}")
print(f"Chunk Size: {chunk_size}, Overlap: {chunk_overlap}, NumRetrieve: {num_retrieve}")
print(f"{'='*60}")

# Create splitter
print("Creating Text Splitter")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

#index docs
print(f"\n{'='*60}")
print("Splitting Documents")
splits = []
for docs in documents_list:
    doc_splits = text_splitter.split_documents(docs)
    for idx, text in enumerate(doc_splits):
      doc_splits[idx].metadata['split_id'] = idx
    print(f'''Number of splits/chunks for {docs}:', {len(doc_splits)}''')
    splits.extend(doc_splits)
print(f"\n{'='*60}")
print(f'''Total number of splits/chunks to load to vector store: {len(splits)}''')


# Adding new chunks to  new vector store
print(f"{'='*60}")
print("Creating new instance of vector store")
qv_store = create_qvector_store(normalized_embeddings, embedding_dim)
print(f"{'='*60}")
print("Adding new splits to  new vector store")
qv_store.add_documents(splits)
retriever = qv_store.as_retriever(search_kwargs={"k": num_retrieve}) # configure number of docs to retrieve
print(f'Completed load to new vector store: {len(splits)}')
print("Loading model and tokenizer outside the langchain config loop")
if use_mistral:
  model_name = LLM_MISTRAL_NAME
  llm_model, llm_tokenizer = load_model_and_tokenizer(model_name)
else:
  model_name = LLM_COHERE_NAME
print("Finished loadting the model and tokenizer")

results = []

if use_mistral:
  llm_name = llm_config['model_name']
  llm_temp = llm_config['llm_temp']
  llm_top_p = llm_config['llm_top_p']
  llm_do_sample = llm_config['llm_do_sample']
  llm_repetition_penalty = llm_config['llm_repetition_penalty']
  llm_max_new_tokens = llm_config['llm_max_new_tokens']
else:
  llm_name = cohere_config['model_name']
  llm_temp = cohere_config['llm_temp']
  llm_top_p = cohere_config['llm_top_p']
  llm_do_sample = cohere_config['llm_do_sample']
  llm_repetition_penalty = cohere_config['llm_repetition_penalty']
  llm_max_new_tokens = cohere_config['llm_max_new_tokens']
if use_mistral:
  llm_lc = configure_llm_pipe(llm_max_new_tokens, llm_model, llm_name, llm_temp, llm_top_p, llm_do_sample, llm_repetition_penalty)
else:
  llm_lc = configure_cohere_llm(llm_max_new_tokens,  llm_temp, llm_top_p, llm_do_sample, llm_repetition_penalty)
for audience_config in audience_configs:
  config_name = audience_config['name']
  rag_prompt = audience_config['prompt']
  audience_type = audience_config['audience_type']
  print(f"{'='*60}")
  print(f"Starting run for : {RAG_EXPERIMENT}, audience_type: {audience_type}, with prompt: {rag_prompt}")
  print(f"{'='*60}")

  # create rag chain
  print(f"\n{'='*60}")
  print("Creating RAG chain")
  rag_prompt_template = get_rag_prompt_template_for_prompt(rag_prompt)
  base_rag_chain = create_rag_chain(audience_type, retriever, rag_prompt_template, llm_lc)

  # execute chain
  print(f"\n{'='*60}")
  print(f'''Running RAG Chain for config: {config_name}''')
  start_time = datetime.now()
  rag_output = base_rag_chain.batch(questions)
  inference_time = (datetime.now() - start_time).total_seconds()
  results.append({
      "rag": rag_output,
      "config_name": config_name,
      "chunk_size": chunk_size,
      "chunk_overlap": chunk_overlap,
      "num_retrieve": num_retrieve,
      "llm_name": llm_name,
      "audience_type": audience_type,
      "embedding_model": emb_model,
      'embedding_dim': embedding_dim,
      'llm_temp': llm_temp,
      'llm_top_p': llm_top_p,
      'llm_do_sample': llm_do_sample,
      'llm_repetition_penalty': llm_repetition_penalty,
      'llm_max_new_tokens': llm_max_new_tokens,
      'inference_time': inference_time
      })
  #save_rag_output(rag_output, chunk_size, chunk_overlap, question_nums, questions,audience_type, llm_name, emb_model, num_retrieve)
  print(f"\n{'='*60}")
  print(f"Completed experiment for config: {config_name}")

  #cleanup
  #qv_store = None
  #retriever = None
  print("Cleaning up before changing audience type")
  rag_prompt_template = None
  base_rag_chain = None
  #llm_lc = None
  print(f"\n{'='*60}")
print(f'''Completed experiments for {RAG_EXPERIMENT} Experiment''')
save_experiment_results(results,question_nums, questions , RAG_EXPERIMENT)
print(f"\n{'='*60}")
print(f" Saved parsed output to {output_folder}rag_{RAG_EXPERIMENT}_outputs.csv")



Output hidden; open in https://colab.research.google.com to view.